# 📊 INSUMOS UNIFICADO — MITRA + PORFIN
### Solo ingresa la fecha y selecciona tus archivos → el notebook hace todo

In [ ]:
# ================================================================
#  CELDA 1 — ÚNICA CELDA QUE DEBES EDITAR
# ================================================================

FECHA_HOY_STR = "20260503"   # ← Solo cambia esto (YYYYMMDD)

# Modo SQL para Mitra (SI / NO)
SQL = "SI"   # ← Cambia si el archivo base de Mitra viene desde SQL

# ¿Qué aplicativos quieres procesar?
PROCESAR_PORFIN = False  # ← False si solo quieres Mitra
PROCESAR_MITRA  = True  # ← False si solo quieres Porfin
# ================================================================

In [ ]:
#  CELDA 2 — IMPORTS Y CONFIGURACIÓN GLOBAL
# ================================================================
%pip install holidays --quiet

import os, re, shutil, glob, time
import pandas as pd
import numpy as np
import holidays
from pathlib import Path
from datetime import datetime, timedelta
import tkinter as tk
from tkinter import filedialog

# Selenium (solo se usa para obtener UVR en Mitra)
try:
    from selenium import webdriver
    from selenium.webdriver.common.by import By
    from selenium.webdriver.chrome.service import Service
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    from webdriver_manager.chrome import ChromeDriverManager
    SELENIUM_OK = True
except ImportError:
    SELENIUM_OK = False
    print("⚠ Selenium no instalado — UVR no se obtendrá automáticamente")

# ── Derivar fechas automáticamente ──────────────────────────────
FECHA_HOY  = datetime.strptime(FECHA_HOY_STR, "%Y%m%d")
FECHA_AYER = FECHA_HOY - timedelta(days=1)
HOY  = FECHA_HOY.strftime("%Y%m%d")
AYER = FECHA_AYER.strftime("%Y%m%d")

# Calendario Colombia (para días hábiles en Mitra)
FESTIVOS_CO = holidays.Colombia()

# ── Rutas comunes ────────────────────────────────────────────────
BASE_INFOVALMER = r"J:\VALORACION\VALORACION_ESPECIAL\Bolsa\INFOVALMER"

# Rutas auxiliares Porfin
RUTA_ESPECIES      = r"C:\Users\dparrado\OneDrive - Skandia Colombia\Valoracion General\MREVISION NEW\Especies.csv"
RUTA_FONDOS_MUTUOS = r"C:\Users\dparrado\OneDrive - Skandia Colombia\Valoracion General\MREVISION NEW\Fondos Mutuos.xlsx"

# Rutas auxiliares Mitra
BASE_MITRA         = r"C:\Users\dparrado\OneDrive - Skandia Colombia\pmitra"
RUTA_FONDOS_MITRA  = rf"{BASE_MITRA}\FONDOS.xlsx"
RUTA_FICS          = rf"{BASE_MITRA}\Indicadores Fics.xlsx"
RUTA_FCPE          = rf"{BASE_MITRA}\FCPE.xlsx"

print(f"📅 HOY  : {HOY}")
print(f"📅 AYER : {AYER}")
print(f"🔧 SQL  : {SQL}")
print(f"📦 Porfin: {'SÍ' if PROCESAR_PORFIN else 'NO'}  |  Mitra: {'SÍ' if PROCESAR_MITRA else 'NO'}")

In [ ]:
#  CELDA 3 — SELECTOR DE ARCHIVOS
#  Ejecuta esta celda para seleccionar los archivos con diálogo.
#  Si prefieres escribir las rutas a mano, puedes saltar esta celda
#  y asignar RUTA_596 y RUTA_ARCHIVO_MITRA directamente.
# ================================================================

def _dialogo(titulo, tipos):
    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)
    ruta = filedialog.askopenfilename(title=titulo, filetypes=tipos,
                                      initialdir=str(Path.home() / "Downloads"))
    root.destroy()
    return ruta

# Archivo 596 (Porfin)
RUTA_596 = None
if PROCESAR_PORFIN:
    RUTA_596 = _dialogo("Selecciona el archivo 596 (Porfin)",
                        [("CSV", "*.csv"), ("Todos", "*.*")])
    print(f"✅ 596 Porfin : {RUTA_596}")

# Archivo base (Mitra)
RUTA_ARCHIVO_MITRA = None
if PROCESAR_MITRA:
    RUTA_ARCHIVO_MITRA = _dialogo("Selecciona el archivo BASE de Mitra",
                                  [("Excel", "*.xlsx *.xls"), ("Todos", "*.*")])
    print(f"✅ Base Mitra : {RUTA_ARCHIVO_MITRA}")

In [ ]:
#  CELDA 4 — UTILIDADES COMPARTIDAS
# ================================================================

# ── Fechas ───────────────────────────────────────────────────────
def sufijo_fecha(fecha_str):
    """20260406 → 040626"""
    return datetime.strptime(fecha_str, "%Y%m%d").strftime("%m%d%y")

def ajustar_a_dia_habil(fecha_str):
    """Retrocede al último día hábil colombiano."""
    fecha = datetime.strptime(fecha_str, "%Y%m%d").date()
    original = fecha
    while fecha.weekday() >= 5 or fecha in FESTIVOS_CO:
        fecha -= timedelta(days=1)
    if fecha != original:
        print(f"⚠ {original} no hábil → usando {fecha}")
    return fecha.strftime("%Y%m%d")

# ── Limpieza numérica ────────────────────────────────────────────
def parse_num_scalar(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().replace("\u00a0", "").replace(" ", "")
    if not s or s.lower() in {"none", "nan", "noaplica", "no aplica"}:
        return np.nan
    s = s.replace(",", "") if ("," in s and "." in s) else s.replace(",", ".")
    s = re.sub(r"[^0-9\.\-]", "", s)
    try:
        return float(s)
    except:
        return np.nan

def parse_num_series(serie):
    return serie.apply(parse_num_scalar)

def parse_date_series(serie):
    return pd.to_datetime(serie, format="%Y%m%d", errors="coerce")

# ── Texto ────────────────────────────────────────────────────────
def clean_text(x):
    return "" if pd.isna(x) else str(x).strip()

def normalize_key(x):
    return "" if pd.isna(x) else re.sub(r"\s+", "", str(x).strip().upper())

def serie_o_vacia(df, col):
    return df[col] if col in df.columns else pd.Series([""] * len(df), index=df.index)

def factor_moneda(valor):
    if pd.isna(valor):
        return 1.0
    s = str(valor).strip().upper()
    if s in {"COP", "$", "COP$", "COL$", "PESOS", "PESO"}:
        return 1.0
    num = parse_num_scalar(s)
    return float(num) if pd.notna(num) else 1.0

# ── Excel helpers ────────────────────────────────────────────────
def excel_ya_existe(carpeta, nombre):
    return os.path.exists(os.path.join(carpeta, f"{nombre}.xlsx"))

def exportar_excel(df, carpeta, nombre):
    os.makedirs(carpeta, exist_ok=True)
    ruta = os.path.join(carpeta, f"{nombre}.xlsx")
    if os.path.exists(ruta):
        print(f"⏭  YA EXISTE → {nombre}.xlsx")
        return ruta
    df.to_excel(ruta, index=False)
    print(f"✔  Generado: {ruta}")
    return ruta

# ── Rutas Infovalmer ─────────────────────────────────────────────
def ruta_dia(fecha_str):
    return os.path.join(BASE_INFOVALMER, fecha_str)

def carpeta_mrevision(fecha_str):
    p = os.path.join(ruta_dia(fecha_str), "MREVISION")
    os.makedirs(p, exist_ok=True)
    return p

# Alias: Porfin guarda en 'Modelo de Rev', Mitra en 'MREVISION'
# Para compatibilidad total ambas carpetas apuntan al mismo lugar.
def carpeta_modelo_rev(fecha_str):
    p = os.path.join(ruta_dia(fecha_str), "Modelo de Rev")
    os.makedirs(p, exist_ok=True)
    return p

print("✅ Utilidades compartidas cargadas")

In [ ]:
#  CELDA 5 — LECTORES INFOVALMER (compartidos)
#  Genera los Excel en MREVISION y en Modelo de Rev para ambos.
# ================================================================

def _fuentes_infovalmer(fecha_str):
    sf = sufijo_fecha(fecha_str)
    base = ruta_dia(fecha_str)
    return {
        "SP"   : os.path.join(base, f"SP{sf}.001"),
        "SW"   : os.path.join(base, f"SW{sf}.001"),
        "MX"   : os.path.join(base, f"MX{sf}.txt"),
        "MX_RV": os.path.join(base, f"MX{sf}_RV.txt"),
        "NOTAS": os.path.join(base, f"NOTAS_ESTRUCTURADAS_{fecha_str}.txt"),
        "TP"   : os.path.join(base, f"titulos_participativos_valoracion_{fecha_str}.txt"),
    }

def _guardar_en_carpetas(df, fecha_str, nombre):
    """Guarda en ambas subcarpetas para compatibilidad Porfin / Mitra."""
    exportar_excel(df, carpeta_mrevision(fecha_str),   nombre)
    exportar_excel(df, carpeta_modelo_rev(fecha_str),  nombre)


def leer_SP(fecha_str):
    nombre = f"SP_{fecha_str}"
    if excel_ya_existe(carpeta_mrevision(fecha_str), nombre):
        return
    fuentes = _fuentes_infovalmer(fecha_str)
    if not os.path.exists(fuentes["SP"]):
        print(f"❌ SP no existe ({fecha_str})")
        return
    datos = []
    with open(fuentes["SP"], "r", encoding="latin-1", errors="ignore") as f:
        for l in f:
            datos.append([
                l[0:7].strip(),  l[7:8].strip(),   l[8:20].strip(),  l[20:32].strip(),
                l[32:50].strip(), l[50:51].strip(), l[51:59].strip(), l[59:67].strip(),
                l[67:75].strip(), l[75:76].strip(), l[76:77].strip(), l[77:81].strip(),
                l[81:84].strip(), l[84:85].strip(), l[85:89].strip(),
                parse_num_scalar(l[89:105]), l[105:109].strip(), l[109:110].strip(),
                parse_num_scalar(l[110:129]), l[129:137].strip(),
                parse_num_scalar(l[137:156]), l[156:168].strip(),
                parse_num_scalar(l[168:187])
            ])
    cols = [
        "Numero_Secuencia","Tipo_Registro","Nemo","ISIN","Numero_Emision",
        "Estado","Fecha_Publicacion","Fecha_Emision","Fecha_Vencimiento",
        "Periodicidad","Modalidad","Dias_Vencimiento","Moneda","Tipo_Tasa",
        "Tipo_Tasa_Ref","Spread","Tipo_Calculo","Base_Calculo",
        "Precio_Valor_Calculado","Fecha_Ultimo_Precio","Ultimo_Precio",
        "Codigo_ISIN2","Precio_Limpio"
    ]
    _guardar_en_carpetas(pd.DataFrame(datos, columns=cols), fecha_str, nombre)


def leer_SW(fecha_str):
    nombre = f"SW_{fecha_str}"
    if excel_ya_existe(carpeta_mrevision(fecha_str), nombre):
        return
    fuentes = _fuentes_infovalmer(fecha_str)
    if not os.path.exists(fuentes["SW"]):
        print(f"❌ SW no existe ({fecha_str})")
        return
    patron = re.compile(r"^(?P<sec>\d+)(?P<tipo>[A-Z])(?P<nemo>[A-Z0-9\-]+)\s+(?P<fecha>\d{4}-\d{2}-\d{2})(?P<valor>.*)$")
    filas = []
    with open(fuentes["SW"], "r", encoding="latin-1", errors="ignore") as f:
        for l in f:
            m = patron.match(l.strip())
            if m:
                val = re.sub(r"[^\d.]", "", m.group("valor"))
                filas.append([m.group("sec"), m.group("tipo"), m.group("nemo"),
                              m.group("fecha"), float(val) if val else None])
    _guardar_en_carpetas(pd.DataFrame(filas, columns=["Secuencia","Tipo","Nemo","Fecha","Valor"]),
                         fecha_str, nombre)


def leer_MX(fecha_str):
    nombre = f"MX_{fecha_str}"
    if excel_ya_existe(carpeta_mrevision(fecha_str), nombre):
        return
    fuentes = _fuentes_infovalmer(fecha_str)
    if os.path.exists(fuentes["MX"]):
        _guardar_en_carpetas(pd.read_csv(fuentes["MX"], dtype=str, encoding="latin-1"),
                             fecha_str, nombre)
    else:
        print(f"❌ MX no existe ({fecha_str})")


def leer_MX_RV(fecha_str):
    nombre = f"MX_RV_{fecha_str}"
    if excel_ya_existe(carpeta_mrevision(fecha_str), nombre):
        return
    fuentes = _fuentes_infovalmer(fecha_str)
    if os.path.exists(fuentes["MX_RV"]):
        _guardar_en_carpetas(pd.read_csv(fuentes["MX_RV"], dtype=str, encoding="latin-1"),
                             fecha_str, nombre)
    else:
        print(f"❌ MX_RV no existe ({fecha_str})")


def leer_NOTAS(fecha_str):
    nombre = f"NOTAS_{fecha_str}"
    if excel_ya_existe(carpeta_mrevision(fecha_str), nombre):
        return
    fuentes = _fuentes_infovalmer(fecha_str)
    if os.path.exists(fuentes["NOTAS"]):
        _guardar_en_carpetas(pd.read_csv(fuentes["NOTAS"], header=None, encoding="latin-1"),
                             fecha_str, nombre)
    else:
        print(f"❌ NOTAS no existe ({fecha_str})")


def leer_TP(fecha_str):
    nombre = f"TP_{fecha_str}"
    if excel_ya_existe(carpeta_mrevision(fecha_str), nombre):
        return
    fuentes = _fuentes_infovalmer(fecha_str)
    if not os.path.exists(fuentes["TP"]):
        print(f"❌ TP no existe ({fecha_str})")
        return
    datos = []
    with open(fuentes["TP"], "r", encoding="latin-1", errors="ignore") as f:
        for l in f:
            if len(l) >= 29:
                datos.append([l[0:6].strip(), l[6:7].strip(), l[7:19].strip(),
                               parse_num_scalar(l[19:29]), l[29:].strip()])
    _guardar_en_carpetas(
        pd.DataFrame(datos, columns=["Codigo","Tipo","ISIN","Precio","Descripcion"]),
        fecha_str, nombre
    )


def copiar_versiones_sp_sw(fecha_str):
    """Copia la versión más reciente de SP/SW a .001 si no existe (para Mitra)."""
    base = ruta_dia(fecha_str)
    sf   = sufijo_fecha(fecha_str)
    if not os.path.exists(base):
        return
    for pref in ["SP", "SW"]:
        dest = os.path.join(base, f"{pref}{sf}.001")
        if os.path.exists(dest):
            continue
        candidatos = sorted(
            [f for f in os.listdir(base) if re.match(rf"{pref}{sf}\.\d+$", f)],
            key=lambda x: int(x.split(".")[-1]), reverse=True
        )
        if candidatos:
            shutil.copy2(os.path.join(base, candidatos[0]), dest)
            print(f"✅ Copiado {candidatos[0]} → {pref}{sf}.001")


def procesar_infovalmer(fecha_str):
    print(f"\n🚀 Infovalmer {fecha_str}")
    copiar_versiones_sp_sw(fecha_str)
    for fn in [leer_SP, leer_SW, leer_MX, leer_MX_RV, leer_NOTAS, leer_TP]:
        try:
            fn(fecha_str)
        except Exception as e:
            print(f"❌ {fn.__name__}: {e}")
    print(f"✅ Infovalmer {fecha_str} listo")


# Procesar AYER y HOY
procesar_infovalmer(AYER)
procesar_infovalmer(HOY)

---
## 🟢 MÓDULO PORFIN

In [ ]:
# ================================================================
#  CELDA 6 — PORFIN: Precios generales (SP + TP + SW)
# ================================================================
if not PROCESAR_PORFIN:
    print("⏭ Módulo Porfin desactivado")
else:

    def _leer_sp_precios(fecha_str):
        path = os.path.join(ruta_dia(fecha_str), f"SP{sufijo_fecha(fecha_str)}.001")
        if not os.path.exists(path):
            return pd.DataFrame(columns=["NEMO","ISIN","PRECIO","PRIORIDAD"])
        df = pd.read_fwf(path, colspecs=[(8,20),(20,32),(110,129)],
                         names=["NEMO","ISIN","PRECIO"], dtype=str)
        df["NEMO"]      = df["NEMO"].apply(normalize_key)
        df["ISIN"]      = df["ISIN"].apply(normalize_key)
        df["PRECIO"]    = parse_num_series(df["PRECIO"])
        df["PRIORIDAD"] = 1
        return df[["NEMO","ISIN","PRECIO","PRIORIDAD"]]

    def _leer_tp_precios(fecha_str):
        path = os.path.join(ruta_dia(fecha_str), f"titulos_participativos_valoracion_{fecha_str}.txt")
        if not os.path.exists(path):
            return pd.DataFrame(columns=["NEMO","ISIN","PRECIO","PRIORIDAD"])
        filas = []
        with open(path, "r", encoding="latin-1", errors="ignore") as f:
            for line in f:
                if len(line.strip()) >= 29:
                    filas.append([normalize_key(line[0:6]), normalize_key(line[7:19]),
                                  parse_num_scalar(line[19:29]), 2])
        return pd.DataFrame(filas, columns=["NEMO","ISIN","PRECIO","PRIORIDAD"])

    def _leer_sw_precios(fecha_str):
        path = os.path.join(ruta_dia(fecha_str), f"SW{sufijo_fecha(fecha_str)}.001")
        if not os.path.exists(path):
            return pd.DataFrame(columns=["NEMO","ISIN","PRECIO","PRIORIDAD"])
        patron = re.compile(r"^(?P<sec>\d+)(?P<tipo>[A-Z])(?P<nemo>[A-Z0-9\-]+)\s+(?P<fecha>\d{4}-\d{2}-\d{2})(?P<valor>.*)$")
        filas = []
        with open(path, "r", encoding="latin-1", errors="ignore") as f:
            for line in f:
                m = patron.match(line.strip())
                if m:
                    nums = re.findall(r"[-+]?\d[\d,\.]*", m.group("valor"))
                    filas.append([normalize_key(m.group("nemo")), "",
                                  parse_num_scalar(nums[-1]) if nums else np.nan, 3])
        return pd.DataFrame(filas, columns=["NEMO","ISIN","PRECIO","PRIORIDAD"])

    def _cargar_precios_generales(fecha_str):
        dfs = [_leer_sp_precios(fecha_str), _leer_tp_precios(fecha_str), _leer_sw_precios(fecha_str)]
        dfs = [d for d in dfs if not d.empty]
        if not dfs:
            return pd.DataFrame(columns=["NEMO","ISIN","PRECIO","PRIORIDAD"])
        df = pd.concat(dfs, ignore_index=True)
        df["PRECIO"] = pd.to_numeric(df["PRECIO"], errors="coerce")
        df["NEMO"]   = df["NEMO"].fillna("").map(normalize_key)
        df["ISIN"]   = df["ISIN"].fillna("").map(normalize_key)
        return df.dropna(subset=["PRECIO"])

    def _construir_mapas(df_precios):
        if df_precios.empty:
            return {}, {}
        df = df_precios.sort_values("PRIORIDAD")
        di = df[df["ISIN"] != ""].drop_duplicates("ISIN", keep="first")
        dn = df[df["NEMO"] != ""].drop_duplicates("NEMO", keep="first")
        return (di.set_index("ISIN")["PRECIO"].to_dict(),
                dn.set_index("NEMO")["PRECIO"].to_dict())

    def _cargar_fondos_mutuos():
        if not os.path.exists(RUTA_FONDOS_MUTUOS):
            return pd.DataFrame(columns=["LLAVE","PRECIO_FM"])
        df = pd.read_excel(RUTA_FONDOS_MUTUOS)
        df.columns = df.columns.astype(str).str.strip()
        lc = next((c for c in df.columns if str(c).upper().strip() in {"LLAVE","KEY","KEYA"}), df.columns[0])
        pc = next((c for c in df.columns if "PRECIO" in str(c).upper()), df.columns[1])
        out = df[[lc, pc]].copy()
        out.columns = ["LLAVE","PRECIO_FM"]
        out["LLAVE"]     = out["LLAVE"].apply(normalize_key)
        out["PRECIO_FM"] = pd.to_numeric(out["PRECIO_FM"].astype(str).str.replace(",","",regex=False), errors="coerce")
        return out.dropna(subset=["PRECIO_FM"]).query("LLAVE != ''").drop_duplicates("LLAVE", keep="first")

    print("✅ Porfin — funciones de precios listas")

In [ ]:
#  CELDA 7 — PORFIN: Motor de valoración
# ================================================================
if not PROCESAR_PORFIN:
    pass
else:

    SPECIAL_LLAVES = {
        "ERBP9" : {"factor": 0.000001,        "tipo": "por_fijo_por"},
        "2ACAC" : {"factor": 180138.96,        "tipo": "nominal"},
        "2FIU1" : {"factor": 9687.34,          "tipo": "nominal"},
        "8LTGK" : {"factor": 1054949999.63,    "tipo": "fijo"},
        "2BVHA" : {"factor": 18368.206568,     "tipo": "vlr_mer_or"},
        "FVBQA" : {"factor": 13593.114169,     "tipo": "vlr_mer_or"},
        **{k: {"factor": v, "tipo": "vlr_mer_or_moneda"} for k, v in {
            "JZCJ3":1.064920,"JZAXO":1.673050,"JZAVN":1.082659,"JHCKT":0.938460,
            "JGDKT":1.160131,"JZAKT":1.076858,"JHPKT":0.958241,"JZAPJ":0.816438,
            "JZDP6":0.845205,"JHM0Y":0.978898,"JZAQK":1.041950,"JZAOK":1.140493,
            "JHHJ2":0.899836,"JHIJ4":1.068267,"JZA1X":1.072365,"JZBJQ":0.999097,
            "JHDJQ":0.946999,"JZANQ":0.984023,"JZEXO":1.638932,"JZAKE":0.943544,
            "JZAY0":1.173927,"JZASS":1.008909,"JHKK0":0.976514,"JZAVX":0.981900,
            "JHS6" :0.885150,"JZAVW":1.080053,"JZFVW":1.079829
        }.items()}
    }

    def calcular_valor(row):
        llave      = normalize_key(row.get("Llave", ""))
        tipo       = clean_text(row.get("TIPO", "")).upper()
        por        = clean_text(row.get("Por", "")).upper()
        nominal    = parse_num_scalar(row.get("Valor Nominal",    np.nan))
        vlr_mer_or = parse_num_scalar(row.get("Valor Mercado Or", np.nan))
        vlr_mer_t  = parse_num_scalar(row.get("Valor Mercado",    np.nan))
        moneda_t   = factor_moneda(row.get("Moned", 1))
        precio_t   = parse_num_scalar(row.get("PRECIO_T",         np.nan))

        if llave in SPECIAL_LLAVES:
            f    = SPECIAL_LLAVES[llave]["factor"]
            modo = SPECIAL_LLAVES[llave]["tipo"]
            if llave == "ERBP9" and por == "92R":     return 106697493950000.00 * 0.000001
            if modo == "fijo":                        return f
            if modo == "nominal":                     return nominal * f
            if modo == "vlr_mer_or":                  return vlr_mer_or * f
            if modo == "vlr_mer_or_moneda":           return vlr_mer_or * f * moneda_t
            if modo == "por_fijo_por":                return 106697493950000.00 * 0.000001

        if tipo in {"FCPE","DFI","FONDOS DE PENSION"}:    return vlr_mer_or * moneda_t
        if tipo == "FM":                                   return vlr_mer_or * moneda_t * precio_t
        if tipo in {"FIC","FCP"}:                          return vlr_mer_or * precio_t
        if tipo in {"CASH","CTA AHORROS","COLATERAL"}:    return nominal * moneda_t
        if tipo in {"ADR","ETF","ACCION INTERNACIONAL"}:  return nominal * moneda_t * precio_t
        if tipo in {"BONO INTERNACIONAL","BONO UVR","CDT UVR","SUBORDINADO INTERNACIONAL",
                    "SUBORDINADO UVR","TES UVR","TIPS","TREASURY","YANKEE","TBILL","TD","NESTR"}:
            return nominal * moneda_t * (precio_t / 100)
        if tipo in {"ACCION","ESTRATEGIAS"}:              return nominal * precio_t
        if tipo in {"BONO","CDT","STRUCTURADO","SUBORDINADO","TES PESOS","TITULARIZADORA","PAPEL COMERCIAL"}:
            return nominal * (precio_t / 100)
        if tipo == "FONDOCRENA":                           return vlr_mer_or * precio_t
        if tipo in {"ANTICIPO","FIDEICOMISO","LOTE","CREDITO","FORWARD"}: return vlr_mer_t
        return vlr_mer_t

    print("✅ Porfin — motor de valoración listo")

In [ ]:
#  CELDA 8 — PORFIN: Pipeline completo 596
# ================================================================
if not PROCESAR_PORFIN:
    pass
else:

    def leer_596(ruta):
        df = pd.read_csv(ruta, sep=";", encoding="latin1",
                         low_memory=False, on_bad_lines="skip", dtype=str)
        df.columns = df.columns.astype(str).str.strip()
        if "Cla" in df.columns:
            df = df[df["Cla"] != "---"].copy()
        df["_ORDEN"] = np.arange(len(df))
        for col in ["ISIN","Nemotécnico Bol","Llave","Moned","Especie","Titulo"]:
            if col in df.columns:
                df[col] = df[col].apply(clean_text)
        for col in ["Emision","F.Vcto","F.Compra"]:
            if col in df.columns:
                df[col] = parse_date_series(df[col])
        for col in ["Valor Nominal","Valor Compra","Valor Mercado","Valor Mercado Or",
                    "Precio","TIR","Por","Vr.Tasa","Margen","T.Dcto","Duración","Dias"]:
            if col in df.columns:
                df[col] = parse_num_series(df[col])
        return df

    def pipeline_porfin(ruta_596, fecha_hoy_str, fecha_ayer_str):
        print("\n" + "="*60)
        print(f"  PORFIN  |  HOY: {fecha_hoy_str}  |  AYER: {fecha_ayer_str}")
        print("="*60)

        print("[1/5] Leyendo 596...")
        df = leer_596(ruta_596)
        original_cols = [c for c in df.columns if c not in {"TIPO","PRECIO_T","PRECIO_Y","VALORACION","_ORDEN"}]

        print("[2/5] Precios HOY...")
        ph = _cargar_precios_generales(fecha_hoy_str)
        mi_h, mn_h = _construir_mapas(ph)

        print("[3/5] Precios AYER...")
        py = _cargar_precios_generales(fecha_ayer_str)
        mi_y, mn_y = _construir_mapas(py)

        fm  = _cargar_fondos_mutuos()
        mfm = fm.set_index("LLAVE")["PRECIO_FM"].to_dict() if not fm.empty else {}

        isin_k  = serie_o_vacia(df, "ISIN").apply(normalize_key)
        nemo_k  = serie_o_vacia(df, "Nemotécnico Bol").apply(normalize_key)
        llave_k = serie_o_vacia(df, "Llave").apply(normalize_key)

        df["PRECIO_T"] = isin_k.map(mi_h)
        df.loc[df["PRECIO_T"].isna(), "PRECIO_T"] = nemo_k[df["PRECIO_T"].isna()].map(mn_h)
        df.loc[llave_k.isin(set(mfm)), "PRECIO_T"] = llave_k[llave_k.isin(set(mfm))].map(mfm)

        df["PRECIO_Y"] = isin_k.map(mi_y)
        df.loc[df["PRECIO_Y"].isna(), "PRECIO_Y"] = nemo_k[df["PRECIO_Y"].isna()].map(mn_y)
        df.loc[llave_k.isin(set(mfm)), "PRECIO_Y"] = llave_k[llave_k.isin(set(mfm))].map(mfm)

        print("[4/5] Tipo de activo...")
        tipos = pd.read_csv(RUTA_ESPECIES, sep=";", encoding="latin1", dtype=str)
        tipos.columns = tipos.columns.astype(str).str.strip()
        tipos["LLAVE"] = tipos["LLAVE"].apply(normalize_key)
        mapa_tipo = tipos.query("LLAVE != ''").drop_duplicates("LLAVE", keep="first").set_index("LLAVE")["TIPO"].to_dict()
        df["TIPO"] = llave_k.map(mapa_tipo)

        print("[5/5] Valoración...")
        df["VALORACION"] = df.apply(calcular_valor, axis=1)

        df = df.sort_values("_ORDEN").drop(columns=["_ORDEN"])
        extras = [c for c in ["TIPO","PRECIO_T","PRECIO_Y","VALORACION"] if c not in original_cols]
        df = df[original_cols + extras]

        salida = os.path.join(Path.home(), "Downloads", f"PORFIN_{fecha_hoy_str}_FINAL.xlsx")
        df.to_excel(salida, index=False)

        print(f"\n✅ Porfin guardado: {salida}")
        print(f"   {len(df):,} filas  |  {df['VALORACION'].notna().sum():,} valorizadas")
        print("="*60)
        return df

    df_porfin = pipeline_porfin(RUTA_596, HOY, AYER)

---
## 🔵 MÓDULO MITRA

In [ ]:
#  CELDA 9 — MITRA: UVR + Monedas + Indicadores FICS
# ================================================================
if not PROCESAR_MITRA:
    print("⏭ Módulo Mitra desactivado")
else:

    FECHA_HABIL = ajustar_a_dia_habil(HOY)
    AYER_HABIL  = ajustar_a_dia_habil(AYER)   # viernes si hoy es lunes
    AYER_UVR    = (datetime.strptime(HOY, "%Y%m%d") - timedelta(days=1)).strftime("%Y%m%d")
    print(f"📅 Fecha hábil HOY       : {FECHA_HABIL}")
    print(f"📅 Fecha hábil AYER      : {AYER_HABIL}  (monedas CSV)")
    print(f"📅 Fecha UVR ayer        : {AYER_UVR}    (UVR Banrep)")

    # ── UVR ──────────────────────────────────────────────────────
    def obtener_uvr(fecha_str):
        if not SELENIUM_OK:
            return None
        fecha_fmt = f"{fecha_str[:4]}/{fecha_str[4:6]}/{fecha_str[6:]}"
        url = "https://suameca.banrep.gov.co/estadisticas-economicas/informacionSerie/100005/unidad_valor_real_uvr"
        opts = Options()
        opts.add_argument("--headless=new")
        opts.add_argument("--window-size=1920,1080")
        opts.add_argument("--disable-gpu")
        opts.add_argument("--no-sandbox")
        opts.add_argument("--disable-dev-shm-usage")
        driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opts)
        try:
            driver.get(url)
            wait = WebDriverWait(driver, 40)
            wait.until(EC.presence_of_element_located((By.TAG_NAME, "app-root")))
            time.sleep(3)
            boton = wait.until(EC.presence_of_element_located((By.ID, "vistaTabla")))
            driver.execute_script("arguments[0].click();", boton)
            wait.until(EC.presence_of_element_located((By.ID, "highcharts-data-table-0")))
            time.sleep(2)
            html = driver.find_element(By.ID, "highcharts-data-table-0").get_attribute("outerHTML")
        finally:
            driver.quit()
        df = pd.read_html(html)[0]
        row = df[df.iloc[:, 0] == fecha_fmt]
        return float(row.iloc[0, 1]) if not row.empty else None

    # ── Monedas ───────────────────────────────────────────────────
    def generar_monedas(fecha_orig, fecha_habil, ayer_habil, uvr_hoy=None, uvr_ayer=None):
        def buscar_csv(fecha):
            for _ in range(30):
                path = os.path.join(ruta_dia(fecha), f"monedas_matriz_info_{fecha}.csv")
                if os.path.exists(path):
                    return path
                fecha = (datetime.strptime(fecha, "%Y%m%d") - timedelta(days=1)).strftime("%Y%m%d")
            return None

        def leer(p):
            df = pd.read_csv(p, sep=None, engine="python", dtype=str)
            df.columns = df.columns.str.strip().str.upper()
            df["RATE"] = pd.to_numeric(df["RATE"], errors="coerce")
            return df

        path_hoy  = buscar_csv(fecha_habil)
        path_ayer = buscar_csv(ayer_habil)

        if not path_hoy:
            raise FileNotFoundError(f"No se encontró monedas_matriz para {fecha_habil}")
        if not path_ayer:
            raise FileNotFoundError(f"No se encontró monedas_matriz para {ayer_habil}")

        df_h = leer(path_hoy)
        df_a = leer(path_ayer)

        mapa = {"USD":"USDCOP","EUR":"EURCOP","MXN":"MXNCOP",
                "BRL":"BRLCOP","GBP":"GBPCOP","JPY":"JPYCOP","CAD":"CADUSD"}
        filas = []
        for m, p in mapa.items():
            h_val = df_h.loc[df_h["PAIR"] == p, "RATE"]
            a_val = df_a.loc[df_a["PAIR"] == p, "RATE"]
            filas.append({"Moneda": m,
                          "Precio Hoy":  h_val.iloc[0] if not h_val.empty else np.nan,
                          "Precio Ayer": a_val.iloc[0] if not a_val.empty else np.nan})

        filas.append({"Moneda": "COP", "Precio Hoy": 1, "Precio Ayer": 1})
        filas.append({"Moneda": "UVR", "Precio Hoy": uvr_hoy, "Precio Ayer": uvr_ayer})

        df_mon = pd.DataFrame(filas)

        # HOY: Precio Hoy = hoy, Precio Ayer = ayer
        exportar_excel(df_mon, carpeta_mrevision(fecha_orig), f"Monedas_{fecha_orig}")

        # ── NUEVO: también genera Monedas_{AYER} para que agregar_monedas(DI) lo encuentre ──
        df_ayer = df_mon.copy()
        df_ayer["Precio Hoy"] = df_mon["Precio Ayer"]   # el "hoy" del viernes = el "ayer" de hoy
        exportar_excel(df_ayer, carpeta_mrevision(AYER), f"Monedas_{AYER}")

        print(f"✅ Monedas generado ({fecha_orig})")
        print(f"✅ Monedas generado ({AYER})  ← para DI")
        print(f"   UVR Hoy  ({HOY})    : {uvr_hoy}")
        print(f"   UVR Ayer ({AYER_UVR}): {uvr_ayer}")

    # ── Indicadores FICS ─────────────────────────────────────────
    def actualizar_indicadores(fecha_orig, fecha_habil):
        meses = {1:"1-ENERO",2:"2-FEBRERO",3:"3-MARZO",4:"4-ABRIL",
                 5:"5-MAYO",6:"6-JUNIO",7:"7-JULIO",8:"8-AGOSTO",
                 9:"9-SEPTIEMBRE",10:"10-OCTUBRE",11:"11-NOVIEMBRE",12:"12-DICIEMBRE"}
        fh = datetime.strptime(fecha_habil, "%Y%m%d").date()
        fo = datetime.strptime(fecha_orig,  "%Y%m%d").date()
        ruta_dia_p = Path(f"J:/VALORACION/PRUEBAS MITRA/Indicadores/{meses[fh.month]}/{fh.day:02d}.xlsx")
        if not ruta_dia_p.exists():
            raise FileNotFoundError(ruta_dia_p)
        df_dia = pd.read_excel(ruta_dia_p)
        df_dia.rename(columns={df_dia.columns[0]: "FECHA"}, inplace=True)
        df_dia["FECHA"] = pd.to_datetime(df_dia["FECHA"], dayfirst=True, errors="coerce").dt.date
        df_dia = df_dia[df_dia["FECHA"] == fh]
        if df_dia.empty:
            raise ValueError("Fecha hábil no encontrada en el archivo de indicadores")
        df_hist = pd.read_excel(RUTA_FICS)
        df_hist["FECHA"] = pd.to_datetime(df_hist["FECHA"], dayfirst=True, errors="coerce").dt.date
        if fo in df_hist["FECHA"].values:
            print("⚠️ Fecha ya existe en histórico")
            return
        for col in set(df_dia.columns) - set(df_hist.columns) - {"FECHA"}:
            df_hist[col] = 0.0
        nueva = {col: 0.0 for col in df_hist.columns}
        nueva["FECHA"] = fo
        for col in set(df_dia.columns) - {"FECHA"}:
            nueva[col] = float(df_dia.iloc[0][col])
        df_hist = pd.concat([df_hist, pd.DataFrame([nueva])], ignore_index=True).sort_values("FECHA")
        df_hist.to_excel(RUTA_FICS, index=False)
        print(f"✅ Indicadores actualizados: {fecha_habil} → {fecha_orig}")

    # ── Ejecutar ──────────────────────────────────────────────────
    print("\n🔄 Obteniendo UVR HOY...")
    try:
        UVR_HOY = obtener_uvr(HOY)
        print(f"💰 UVR HOY  ({HOY})    : {UVR_HOY}")
    except Exception as e:
        UVR_HOY = None
        print(f"❌ UVR HOY: {e}")

    print("\n🔄 Obteniendo UVR AYER...")
    try:
        UVR_AYER = obtener_uvr(AYER_UVR)
        print(f"💰 UVR AYER ({AYER_UVR}): {UVR_AYER}")
    except Exception as e:
        UVR_AYER = None
        print(f"❌ UVR AYER: {e}")

    print("\n🔄 Generando monedas...")
    try:
        generar_monedas(HOY, FECHA_HABIL, AYER_HABIL,
                        uvr_hoy=UVR_HOY, uvr_ayer=UVR_AYER)
    except Exception as e:
        print(f"❌ Monedas: {e}")

    print("\n🔄 Actualizando indicadores FICS...")
    try:
        actualizar_indicadores(HOY, FECHA_HABIL)
    except Exception as e:
        print(f"❌ Indicadores FICS: {e}")

In [ ]:
#  CELDA 10 — MITRA: Carga del archivo base (p1)
# ================================================================
if not PROCESAR_MITRA:
    pass
else:
    import openpyxl

    def _encontrar_hoja_datos(ruta):
        """
        Encuentra la hoja con datos reales: la que tenga más columnas
        en su primera fila y no sea solo XML de parámetros.
        """
        wb = openpyxl.load_workbook(ruta, read_only=True, data_only=True)
        mejor_hoja  = None
        mejor_score = 0

        for nombre in wb.sheetnames:
            ws = wb[nombre]
            filas = list(ws.iter_rows(max_row=2, values_only=True))
            if not filas:
                continue
            fila0 = filas[0]
            # Descartar hojas cuya primera celda sea XML/schema
            primera_celda = str(fila0[0] or "")
            if any(kw in primera_celda for kw in ("<xs:", "<Parametros>", "xs:schema", "_x000D_")):
                continue
            # Score = número de celdas no nulas en la primera fila
            score = sum(1 for v in fila0 if v is not None)
            if score > mejor_score:
                mejor_score = score
                mejor_hoja  = nombre

        wb.close()
        if mejor_hoja is None:
            raise ValueError(
                f"No se encontró ninguna hoja con datos tabulares en:\n{ruta}\n"
                "Hojas disponibles: " + str(openpyxl.load_workbook(ruta, read_only=True).sheetnames)
            )
        return mejor_hoja

    hoja = _encontrar_hoja_datos(RUTA_ARCHIVO_MITRA)
    print(f"ℹ️ Hoja seleccionada: '{hoja}'")

    p1 = pd.read_excel(RUTA_ARCHIVO_MITRA, sheet_name=hoja, engine="openpyxl")
    p1.columns = [c.upper().strip() for c in p1.columns]

    if SQL == "SI":
        p1 = p1.rename(columns={
            'TIPO_PROD'               : 'TIPO DE PRODUCTO',
            'PRODUCTO'                : 'NÍVEL 3 - PRODUCTO',
            'CARTEIRA'                : 'NÍVEL  2 - CARTERA',
            'ISIN'                    : 'CÓDIGO ISIN CONTRATO',
            'MONEDA'                  : 'MONEDA PRODUCTO',
            'METODOLOGIA'             : 'MARCADO ACTIVO',
            'VRNOMINAL_DI'            : 'CANTIDAD_DI',
            'VALOR_MDO_ACTIVO_INI'    : 'FINANCIERO ACTIVO_DI',
            'VALOR_MDO_PASIVO_INI'    : 'FINANCIERO PASIVO_DI',
            'VALOR_MDO_INI'           : 'POSICION_DI',
            'VALOR_MDO_ALAVANCADO_INI': 'FINANCIERO ALAVANCADO_DI',
            'VRNOMINAL_DF'            : 'CANTIDAD_DF',
            'VALOR_MDO_ACTIVO_FIN'    : 'FINANCIERO ACTIVO_DF',
            'VALOR_MDO_PASIVO_FIN'    : 'FINANCIERO PASIVO_DF',
            'VALOR_MDO_FIN'           : 'POSICION_DF',
            'VALOR_MDO_ALAVANCADO_FIN': 'FINANCIERO ALAVANCADO_DF',
            'VRNOMINAL_RESIDUAL_DI'   : 'VALOR NOMINAL REMANESCENTE (MOEDA PRODUTO)_DI',
            'VRNOMINAL_RESIDUAL_DF'   : 'VALOR NOMINAL REMANESCENTE (MOEDA PRODUTO)_DF',
            'CAUSACION_TOTAL'         : 'DI - DF',
            'PROVISIÓN_DI'            : 'PROVISION_DI',
            'PROVISIÓN_DF'            : 'PROVISION_DF',
            'POSICIÓN_NETA'           : 'POSICION_NETA',
            'VARIACIÓN_PROVISIÓN'     : 'VARIACION_PROVISION',
        })
    else:
        primera_fila = p1.iloc[0].astype(str).str.upper().str.strip().tolist()
        nombres_col  = [c.upper().strip() for c in p1.columns.astype(str).tolist()]
        if primera_fila == nombres_col:
            print("⚠️ Header repetido en fila 0 → eliminado")
            p1 = p1.iloc[1:].reset_index(drop=True)
        else:
            p1.columns = p1.iloc[0].astype(str).str.upper().str.strip()
            p1 = p1.iloc[1:].reset_index(drop=True)

        p1 = p1.rename(columns={
            'NÍVEL 3 - PRODUTO':   'NÍVEL 3 - PRODUCTO',
            'NÍVEL  2 - CARTEIRA': 'NÍVEL  2 - CARTERA',
        })

    # Buscar TIPO DE PRODUCTO de forma robusta
    col_tipo = next(
        (c for c in p1.columns
         if re.sub(r'[\s_]+', '', c.upper())
            in {'TIPODEPRODUCTO', 'TIPOPROD', 'TIPODEPRODUTO'}),
        None
    )
    if col_tipo is None:
        print("⚠️ No se encontró columna TIPO DE PRODUCTO")
        print("   Columnas disponibles:", list(p1.columns))
    else:
        if col_tipo != 'TIPO DE PRODUCTO':
            p1 = p1.rename(columns={col_tipo: 'TIPO DE PRODUCTO'})
        patron_excl = 'Moeda|Swap|Futuro'
        p1 = p1[~p1['TIPO DE PRODUCTO'].astype(str).str.contains(
            patron_excl, case=False, na=False
        )].reset_index(drop=True)

    print(f"✅ Mitra p1 cargado: {len(p1):,} filas, {len(p1.columns)} columnas")

In [ ]:
#  CELDA 11 — MITRA: Pipeline de enriquecimiento de precios
# ================================================================
if not PROCESAR_MITRA:
    pass
else:

    def _mr(fecha_str):
        return carpeta_mrevision(fecha_str)

    def _norm(x):
        """Normaliza llaves para evitar problemas por tipos (int/float/str), espacios y .0 de Excel."""
        if pd.isna(x):
            return None
        s = str(x).strip().upper()
        if s.endswith(".0"):
            s = s[:-2]
        return s

    def _asegurar_columna(df, col, valor_inicial=pd.NA):
        if col not in df.columns:
            df[col] = valor_inicial
        return df

    def _mapa_unico(df_ref, clave_ref, col_precio):
        """
        Crea un mapa 1 a 1: clave -> precio
        Si hay repetidos en la referencia, se queda con el primero.
        """
        if df_ref.empty:
            return pd.Series(dtype="object")

        tmp = df_ref[[clave_ref, col_precio]].copy()
        tmp[clave_ref] = tmp[clave_ref].map(_norm)
        tmp = tmp[tmp[clave_ref].notna()]
        tmp = tmp.drop_duplicates(subset=[clave_ref], keep="first")
        return tmp.set_index(clave_ref)[col_precio]

    def _aplicar_mapa(df, col_dest, clave_df, mapa, fuente):
        """
        Rellena SOLO las filas faltantes de col_dest usando map.
        Nunca cambia el número de filas.
        """
        _asegurar_columna(df, col_dest, pd.NA)
        _asegurar_columna(df, "FUENTE_PRECIO", pd.NA)

        faltantes = df[col_dest].isna()
        if not faltantes.any() or mapa.empty:
            return df

        llaves = df.loc[faltantes, clave_df].map(_norm)
        valores = llaves.map(mapa)

        df.loc[faltantes, col_dest] = valores.values

        nuevos = faltantes & df[col_dest].notna()
        df.loc[nuevos, "FUENTE_PRECIO"] = fuente

        return df

    def enriquecer_sp(df, hoy, ayer):
        col_isin = "ISIN"
        col_prec = "PRECIO_VALOR_CALCULADO"
        clave_p1 = "CÓDIGO ISIN CONTRATO"

        _asegurar_columna(df, "FUENTE_PRECIO", pd.NA)
        _asegurar_columna(df, "PRECIO_T", pd.NA)
        _asegurar_columna(df, "PRECIO_Y", pd.NA)

        for fecha, col_dest in [(hoy, "PRECIO_T"), (ayer, "PRECIO_Y")]:
            try:
                src = pd.read_excel(os.path.join(_mr(fecha), f"SP_{fecha}.xlsx"))
                src.columns = [str(c).upper().strip() for c in src.columns]

                src = src[[col_isin, col_prec]].copy()
                src["CLAVE"] = src[col_isin].map(_norm)
                src["PRECIO"] = src[col_prec]

                mapa = _mapa_unico(src, "CLAVE", "PRECIO")
                df = _aplicar_mapa(df, col_dest, clave_p1, mapa, "SP")

            except Exception as e:
                print(f"⚠ SP {fecha}: {e}")

        print("✅ SP")
        return df

    def enriquecer_fcpe(df, hoy):
        try:
            fcpe = pd.read_excel(RUTA_FCPE, sheet_name="2025", header=4)
            fcpe.columns = [str(c).upper().strip() for c in fcpe.columns]
        except Exception as e:
            print(f"⚠ FCPE: {e}")
            return df

        fecha_dt = datetime.strptime(hoy, "%Y%m%d")

        _asegurar_columna(df, "FUENTE_PRECIO", pd.NA)
        _asegurar_columna(df, "PRECIO_T", pd.NA)
        _asegurar_columna(df, "PRECIO_Y", pd.NA)

        if "NOMBRE MITRA" not in fcpe.columns:
            print("❌ FCPE sin columna NOMBRE MITRA")
            return df

        fcpe["CLAVE_FCPE"] = fcpe["NOMBRE MITRA"].map(_norm)

        pcols = sorted([c for c in fcpe.columns if c.startswith("PRECIO_Q")])
        fcols = sorted([c for c in fcpe.columns if c.startswith("F. REGISTRO_Q")])

        fcpe["PRECIO_VALIDO"] = None

        for idx, row in fcpe.iterrows():
            mejores = []
            for fc, pc in zip(fcols, pcols):
                if pd.isna(row[fc]):
                    continue
                try:
                    fq = pd.to_datetime(row[fc], dayfirst=True)
                except:
                    continue
                if fq <= fecha_dt:
                    mejores.append((fq, row[pc]))
            if mejores:
                fcpe.at[idx, "PRECIO_VALIDO"] = sorted(mejores, key=lambda x: x[0], reverse=True)[0][1]

        mapa = _mapa_unico(fcpe, "CLAVE_FCPE", "PRECIO_VALIDO")

        for col in ["PRECIO_T", "PRECIO_Y"]:
            df = _aplicar_mapa(df, col, "NÍVEL 3 - PRODUCTO", mapa, "FCPE")

        print("✅ FCPE")
        return df

    def enriquecer_mx_rv(df, hoy, ayer):
        _asegurar_columna(df, "FUENTE_PRECIO", pd.NA)
        _asegurar_columna(df, "PRECIO_T", pd.NA)
        _asegurar_columna(df, "PRECIO_Y", pd.NA)

        moneda_norm = df["MONEDA PRODUCTO"].map(_norm).fillna("").str.replace(r"COP\.INF$", "", regex=True)
        df["CLAVE_P1"] = df["CÓDIGO ISIN CONTRATO"].map(_norm).fillna("") + moneda_norm

        for fecha, col_dest in [(hoy, "PRECIO_T"), (ayer, "PRECIO_Y")]:
            try:
                src = pd.read_excel(os.path.join(_mr(fecha), f"MX_RV_{fecha}.xlsx"))
                src.columns = [str(c).upper().strip() for c in src.columns]

                src["CLAVE"] = src["ISIN"].map(_norm).fillna("") + src["MONEDA"].map(_norm).fillna("")
                src["PRECIO"] = src["PRECIO"]

                mapa = _mapa_unico(src, "CLAVE", "PRECIO")
                df = _aplicar_mapa(df, col_dest, "CLAVE_P1", mapa, "MX_RV")

            except Exception as e:
                print(f"⚠ MX_RV {fecha}: {e}")

        df.drop(columns=["CLAVE_P1"], inplace=True, errors="ignore")
        print("✅ MX_RV")
        return df

    def enriquecer_mx(df, hoy, ayer):
        _asegurar_columna(df, "FUENTE_PRECIO", pd.NA)
        _asegurar_columna(df, "PRECIO_T", pd.NA)
        _asegurar_columna(df, "PRECIO_Y", pd.NA)

        df["CLAVE_P1"] = df["CÓDIGO ISIN CONTRATO"].map(_norm).fillna("") + df["MONEDA PRODUCTO"].map(_norm).fillna("")

        for fecha, col_dest in [(hoy, "PRECIO_T"), (ayer, "PRECIO_Y")]:
            try:
                src = pd.read_excel(os.path.join(_mr(fecha), f"MX_{fecha}.xlsx"))
                src.columns = [str(c).upper().strip() for c in src.columns]

                if "PRECIO SUCIO" not in src.columns:
                    continue

                src["CLAVE"] = src["ISIN"].map(_norm).fillna("") + src["MONEDA"].map(_norm).fillna("")
                src["PRECIO"] = src["PRECIO SUCIO"]

                mapa = _mapa_unico(src, "CLAVE", "PRECIO")
                df = _aplicar_mapa(df, col_dest, "CLAVE_P1", mapa, "MX")

            except Exception as e:
                print(f"⚠ MX {fecha}: {e}")

        df.drop(columns=["CLAVE_P1"], inplace=True, errors="ignore")
        print("✅ MX")
        return df

    def enriquecer_sw(df, hoy, ayer):
        _asegurar_columna(df, "FUENTE_PRECIO", pd.NA)
        _asegurar_columna(df, "PRECIO_T", pd.NA)
        _asegurar_columna(df, "PRECIO_Y", pd.NA)

        df["CLAVE_P1"] = df["NÍVEL 3 - PRODUCTO"].map(_norm).fillna("").str.replace(r"\.BVC$", "", regex=True)

        for fecha, col_dest in [(hoy, "PRECIO_T"), (ayer, "PRECIO_Y")]:
            try:
                src = pd.read_excel(os.path.join(_mr(fecha), f"SW_{fecha}.xlsx"))
                src.columns = [str(c).upper().strip() for c in src.columns]

                src["CLAVE"] = src["NEMO"].map(_norm)
                src["PRECIO"] = src["VALOR"]

                mapa = _mapa_unico(src, "CLAVE", "PRECIO")
                df = _aplicar_mapa(df, col_dest, "CLAVE_P1", mapa, "SW")

            except Exception as e:
                print(f"⚠ SW {fecha}: {e}")

        df.drop(columns=["CLAVE_P1"], inplace=True, errors="ignore")
        print("✅ SW")
        return df

    def enriquecer_notas(df, hoy, ayer):
        _asegurar_columna(df, "FUENTE_PRECIO", pd.NA)
        _asegurar_columna(df, "PRECIO_T", pd.NA)
        _asegurar_columna(df, "PRECIO_Y", pd.NA)

        df["CLAVE_P1"] = df["NÍVEL 3 - PRODUCTO"].map(_norm)

        for fecha, col_dest in [(hoy, "PRECIO_T"), (ayer, "PRECIO_Y")]:
            try:
                src = pd.read_excel(os.path.join(_mr(fecha), f"NOTAS_{fecha}.xlsx"))
                src.columns = [str(c).upper().strip() for c in src.columns]

                src["CLAVE"] = src["ISIN2"].map(_norm)
                src["PRECIO"] = src["PRECIO SUCIO"]

                mapa = _mapa_unico(src, "CLAVE", "PRECIO")
                df = _aplicar_mapa(df, col_dest, "CLAVE_P1", mapa, "NE")

            except Exception as e:
                print(f"⚠ NOTAS {fecha}: {e}")

        df.drop(columns=["CLAVE_P1"], inplace=True, errors="ignore")
        print("✅ Notas")
        return df

    def enriquecer_tp(df, hoy, ayer):
        _asegurar_columna(df, "FUENTE_PRECIO", pd.NA)
        _asegurar_columna(df, "PRECIO_T", pd.NA)
        _asegurar_columna(df, "PRECIO_Y", pd.NA)

        df["CLAVE_P1"] = df["CÓDIGO ISIN CONTRATO"].map(_norm)

        for fecha, col_dest in [(hoy, "PRECIO_T"), (ayer, "PRECIO_Y")]:
            try:
                src = pd.read_excel(os.path.join(_mr(fecha), f"TP_{fecha}.xlsx"))
                src.columns = [str(c).upper().strip() for c in src.columns]

                src["CLAVE"] = src["ISIN"].map(_norm)
                src["PRECIO"] = src["DESCRIPCION"]

                mapa = _mapa_unico(src, "CLAVE", "PRECIO")
                df = _aplicar_mapa(df, col_dest, "CLAVE_P1", mapa, "TP")

            except Exception as e:
                print(f"⚠ TP {fecha}: {e}")

        df.drop(columns=["CLAVE_P1"], inplace=True, errors="ignore")
        print("✅ TP")
        return df

    def enriquecer_fics(df, hoy, ayer):
        try:
            fics = pd.read_excel(RUTA_FICS, sheet_name="Sheet1")
            fics.columns = [str(c).upper().strip() for c in fics.columns]
        except Exception as e:
            print(f"⚠ FICS: {e}")
            return df

        def conv_fecha(x):
            try:
                if isinstance(x, datetime):
                    return x.strftime("%Y%m%d")
                return datetime.strptime(str(x), "%d/%m/%Y").strftime("%Y%m%d")
            except:
                return None

        _asegurar_columna(df, "FUENTE_PRECIO", pd.NA)
        _asegurar_columna(df, "PRECIO_T", pd.NA)
        _asegurar_columna(df, "PRECIO_Y", pd.NA)

        fics["FECHA"] = fics["FECHA"].apply(conv_fecha)
        fondos = [c for c in fics.columns if c != "FECHA"]

        df["CLAVE_P1"] = df["NÍVEL 3 - PRODUCTO"].map(_norm)

        for fecha, col_dest in [(hoy, "PRECIO_T"), (ayer, "PRECIO_Y")]:
            fila = fics.loc[fics["FECHA"] == fecha, fondos]

            if fila.empty:
                print(f"⚠ FICS sin datos para {fecha}")
                continue

            # Evita el error de int.upper()
            mapeo = {}
            for k, v in fila.iloc[0].to_dict().items():
                kk = _norm(k)
                if kk is not None and kk != "FECHA":
                    mapeo[kk] = v

            faltan = df[col_dest].isna()
            if faltan.any():
                valores = df.loc[faltan, "CLAVE_P1"].map(mapeo)
                df.loc[faltan, col_dest] = valores.values
                df.loc[faltan & df[col_dest].notna(), "FUENTE_PRECIO"] = "FICS"

        df.drop(columns=["CLAVE_P1"], inplace=True, errors="ignore")
        print("✅ FICS")
        return df

    def agregar_monedas(df, fecha, sufijo):
        ruta = os.path.join(_mr(fecha), f"Monedas_{fecha}.xlsx")

        try:
            dfm = pd.read_excel(ruta)
            dfm.columns = [str(c).upper().strip() for c in dfm.columns]

            # Normaliza y evita duplicados en la tabla de referencia
            dfm["MONEDA_N"] = dfm["MONEDA"].map(_norm)
            if "PRECIO HOY" not in dfm.columns:
                print(f"⚠ Monedas {sufijo}: no existe la columna PRECIO HOY")
                return df

            dfm = dfm.drop_duplicates(subset=["MONEDA_N"], keep="first")
            mapa = dfm.set_index("MONEDA_N")["PRECIO HOY"]

            col_nueva = f"MONEDA_{sufijo}"
            df[col_nueva] = df["MONEDA PRODUCTO"].map(_norm).map(mapa)

            print(f"✅ Monedas {sufijo}")

        except Exception as e:
            print(f"⚠ Monedas {sufijo}: {e}")

        return df

    # ── Ejecutar pipeline ─────────────────────────────────────────
    print("\n" + "=" * 60)
    print(f"  MITRA  |  HOY: {HOY}  |  AYER: {AYER}")
    print("=" * 60)

    filas_iniciales = len(p1)
    
    
    from datetime import datetime, timedelta

    AYERMONEDA = (datetime.strptime(AYER, "%Y%m%d") - timedelta(days=2)).strftime("%Y%m%d")


    p1 = enriquecer_sp(p1, HOY, AYER)
    p1 = enriquecer_fcpe(p1, HOY)
    p1 = enriquecer_mx_rv(p1, HOY, AYER)
    p1 = enriquecer_mx(p1, HOY, AYER)
    p1 = enriquecer_sw(p1, HOY, AYER)
    p1 = agregar_monedas(p1, HOY,  "DF")
    p1 = agregar_monedas(p1, AYER, "DI")
    p1 = enriquecer_notas(p1, HOY, AYER)
    p1 = enriquecer_tp(p1, HOY, AYER)
    p1 = enriquecer_fics(p1, HOY, AYER)

    print(f"\nFilas iniciales: {filas_iniciales:,}")
    print(f"Filas finales:   {len(p1):,}")

    if len(p1) != filas_iniciales:
        print("⚠ Aviso: el número de filas cambió. Con este enfoque no debería pasar; revisa si p1 ya traía duplicados antes del pipeline.")

    print(f"\n✅ Enriquecimiento completado: {len(p1):,} filas")


In [ ]:
#  CELDA 12 — MITRA: Cálculos, validaciones y causación
# ================================================================
if not PROCESAR_MITRA:
    pass
else:

    def calcular_mitra(p1):
        # Normalizar columnas duplicadas con sufijos _DI / _DF
        cols_norm = [
            c.strip().upper().replace("Ó","O").replace("Í","I")
                              .replace("Á","A").replace("É","E").replace("Ú","U")
            for c in p1.columns
        ]
        conteo = pd.Series(cols_norm).value_counts()
        usados, new_cols = {}, []
        for col in cols_norm:
            if conteo[col] == 1:
                new_cols.append(col)
            else:
                usados[col] = usados.get(col, 0) + 1
                new_cols.append(f"{col}_{'DI' if usados[col]==1 else 'DF'}")
        p1.columns = new_cols

        num_cols = [
            "MONEDA","MONEDA_DI","MONEDA_DF","PRECIO","PRECIO_T","PRECIO_Y",
            "CANTIDAD_DI","CANTIDAD_DF","POSICION_DI","POSICION_DF",
            "VALORACION_DI","VALORACION_DF","FINANCIERO ACTIVO","FINANCIERO ACTIVO_DI",
            "VALOR NOMINAL REMANESCENTE (MOEDA PRODUTO)",
            "VALOR NOMINAL REMANESCENTE (MOEDA PRODUTO)_DI",
            "VALOR NOMINAL REMANESCENTE (MOEDA PRODUTO)_DF"
        ]
        for c in num_cols:
            if c in p1.columns:
                p1[c] = pd.to_numeric(
                    p1[c].astype(str).str.replace(",","",regex=False).str.strip(),
                    errors="coerce"
                )

        rf = p1["TIPO DE PRODUCTO"] == "Renda Fixa"
        p1["VALORACION_DI"] = np.nan
        p1["VALORACION_DF"] = np.nan

        p1.loc[rf,  "VALORACION_DI"] = p1["MONEDA_DI"] / 100 * p1["PRECIO_Y"] * p1["VALOR NOMINAL REMANESCENTE (MOEDA PRODUTO)_DI"]
        p1.loc[rf,  "VALORACION_DF"] = p1["MONEDA_DF"] / 100 * p1["PRECIO_T"] * p1["VALOR NOMINAL REMANESCENTE (MOEDA PRODUTO)_DF"]
        p1.loc[~rf, "VALORACION_DI"] = p1["MONEDA_DI"] * p1["PRECIO_Y"] * p1["FINANCIERO ACTIVO_DI"]
        p1.loc[~rf, "VALORACION_DF"] = p1["MONEDA_DF"] * p1["PRECIO_T"] * p1["FINANCIERO ACTIVO_DF"]

        p1["V_VALORACION_DI"] = np.where((p1["POSICION_DI"]-p1["VALORACION_DI"]).abs() < 0.5, 0,
                                          (p1["POSICION_DI"]-p1["VALORACION_DI"]).round(3))
        p1["V_VALORACION_DF"] = np.where((p1["POSICION_DF"]-p1["VALORACION_DF"]).abs() < 0.5, 0,
                                          (p1["POSICION_DF"]-p1["VALORACION_DF"]).round(3))

        p1["OBSERVACION"] = ""
        p1.loc[(p1["V_VALORACION_DI"] != 0) & (p1["V_VALORACION_DI"].abs() > 0.5) &
               (p1["MARCADO ACTIVO"] == "Curva"), "OBSERVACION"] = "Titulo a tir"
        var_precio = (p1["PRECIO_T"] - p1["PRECIO_Y"]).abs() / p1["PRECIO_Y"]
        p1.loc[var_precio > 0.05, "OBSERVACION"] = "Variacion en precio"

        cond = p1["VALORACION_DF"].isna() & p1["VALORACION_DI"].notna()
        p1.loc[cond & rf,  "VALORACION_DF"] = p1["FINANCIERO ACTIVO_DF"] * p1["PRECIO_T"] / 100
        p1.loc[cond & ~rf, "VALORACION_DF"] = p1["FINANCIERO ACTIVO_DF"] * p1["PRECIO_T"]
        p1.loc[cond, "OBSERVACION"] = "Titulo primario"

        p1["CAUSACION"] = p1["VALORACION_DF"] - p1["VALORACION_DI"]
        return p1

    p1 = calcular_mitra(p1)
    print(f"✅ Mitra cálculos completados: {len(p1):,} filas")

In [ ]:
#leer 575 Fundaciones
RUTA_575 = rf"J:\VALORACION\PRUEBAS MITRA\Marcha blanca fiduciaria\575fun{HOY}.CSV"
caus = pd.read_csv(RUTA_575, sep=";", encoding="cp1252")
caus.columns = caus.columns.str.strip()

caus.columns

p1["TITULO"] = p1["TITULO"].astype(str).str.strip()
caus["Título"] = caus["Título"].astype(str).str.strip()

caus = caus.rename(columns={
    "Vlr Nominal": "Nominal Porfin",
    "Vlr Mer. Ant": "Porfin Ant",
    "Vlr Mer. Hoy": "Porfin Hoy",
    "Causación Mer": "Causación Mer Porfin"
})

cols_porfin = [
    "Nominal Porfin",
    "Porfin Ant",
    "Porfin Hoy",
    "Causación Mer Porfin"
]

p1 = p1.drop(columns=[c for c in cols_porfin if c in p1.columns], errors="ignore")

p1 = p1.merge(
    caus[[
        "Título",
        "Nominal Porfin",
        "Porfin Ant",
        "Porfin Hoy",
        "Causación Mer Porfin"
    ]],
    left_on="TITULO",
    right_on="Título",
    how="left"
)

p1

In [ ]:
#  CELDA 13 — MITRA: Consolidar causaciones Porfin + merge fondos
# ================================================================
if not PROCESAR_MITRA:
    pass
else:
    import unicodedata

    def limpiar_cols(df):
        df.columns = (
            df.columns.astype(str).str.strip()
            .str.replace("\n", " ", regex=True)
            .str.replace("  ", " ", regex=True)
            .str.replace(".", "", regex=False)
            .str.replace(" ", "", regex=False)
        )
        return df

    def _canon(txt):
        if pd.isna(txt):
            return ""
        txt = str(txt).strip().upper()
        txt = unicodedata.normalize("NFKD", txt).encode("ascii", "ignore").decode("ascii")
        txt = re.sub(r"[^A-Z0-9]+", "", txt)
        return txt

    def _find_col(df, candidates):
        cmap = {_canon(c): c for c in df.columns}
        for cand in candidates:
            key = _canon(cand)
            if key in cmap:
                return cmap[key]
        raise KeyError(f"No encontré ninguna columna entre: {candidates}")

    def _norm_txt(x):
        if pd.isna(x):
            return ""
        s = str(x).strip()
        if s.endswith(".0"):
            s = s[:-2]
        return s

    def _norm_key(x):
        return _canon(_norm_txt(x))

    # ============================================================
    # Fecha operativa:
    # - Causaciones: HOY
    # - Cupos 596: HOY_CAUS (viernes -> domingo)
    # ============================================================
    fecha_dt = datetime.strptime(HOY, "%Y%m%d")

    HOY_CAUS = HOY
    if fecha_dt.weekday() == 4:      # viernes
        HOY_CAUS = (fecha_dt + timedelta(days=2)).strftime("%Y%m%d")
    elif fecha_dt.weekday() == 5:    # sábado
        HOY_CAUS = (fecha_dt + timedelta(days=1)).strftime("%Y%m%d")

    # ============================================================
    # Causaciones Porfin
    # ============================================================
    ruta_caus = rf"N:\Valoracion-Informes\Causaciones\{HOY}"

    porfin_caus = None

    try:
        archivos = glob.glob(os.path.join(ruta_caus, "*.xlsx"))
        dfs = []

        for a in archivos:
            try:
                d = pd.read_excel(a, engine="openpyxl")
                d = limpiar_cols(d)

                if d.empty:
                    continue

                d = d[d.iloc[:, 0].notna()]
                d = d.dropna(axis=1, how="all")
                d = d.dropna(thresh=max(1, d.shape[1] - 4))

                d["ARCHIVO_ORIGEN"] = os.path.basename(a)
                dfs.append(d)

            except Exception as e:
                print(f"⚠ {a}: {e}")

        if dfs:
            porfin_caus = pd.concat(dfs, ignore_index=True)
            porfin_caus = porfin_caus.loc[:, ~porfin_caus.columns.duplicated()].copy()

            # ========================================================
            # PILAS (596) -> usa HOY_CAUS
            # ========================================================
            ruta_pilas = rf"N:\Valoracion-Informes\Cupos\{HOY_CAUS}\596{HOY_CAUS}.CSV"

            pilas = pd.read_csv(
                ruta_pilas,
                encoding="latin1",
                sep=";",
                skip_blank_lines=True,
                engine="python"
            )
            pilas = limpiar_cols(pilas)

            col_pilas_titulo = _find_col(pilas, ["Titulo", "Título", "TITULO"])
            col_pilas_isin   = _find_col(pilas, ["ISIN"])

            pilas[col_pilas_titulo] = pilas[col_pilas_titulo].astype(str).str.replace(".0", "", regex=False).str.strip()
            pilas[col_pilas_isin]   = pilas[col_pilas_isin].astype(str).str.replace(".0", "", regex=False).str.strip()

            col_caus_titulo = _find_col(porfin_caus, ["Título", "Titulo", "TITULO"])
            porfin_caus[col_caus_titulo] = porfin_caus[col_caus_titulo].astype(str).str.replace(".0", "", regex=False).str.strip()

            # Mapa Título -> ISIN desde PILAS, sin multiplicar filas
            pilas_tmp = (
                pilas[[col_pilas_titulo, col_pilas_isin]]
                .dropna(subset=[col_pilas_titulo])
                .drop_duplicates(subset=[col_pilas_titulo], keep="first")
                .copy()
            )
            pilas_tmp["TITULO_STD"] = pilas_tmp[col_pilas_titulo].map(_norm_key)
            mapa_isin_pilas = pilas_tmp.set_index("TITULO_STD")[col_pilas_isin]

            porfin_caus["TITULO_STD"] = porfin_caus[col_caus_titulo].map(_norm_key)

            if "ISIN" not in porfin_caus.columns:
                porfin_caus["ISIN"] = pd.NA

            porfin_caus["ISIN"] = porfin_caus["ISIN"].fillna(porfin_caus["TITULO_STD"].map(mapa_isin_pilas))

            # Columna valor de causación
            col_valor_caus = _find_col(porfin_caus, ["VALORPORFIN", "VALOR PORFIN", "VALOR"])
            porfin_caus[col_valor_caus] = pd.to_numeric(porfin_caus[col_valor_caus], errors="coerce").fillna(0)

            ruta_salida_porfin = os.path.join(carpeta_mrevision(HOY), f"porfin_{HOY}.xlsx")
            os.makedirs(os.path.dirname(ruta_salida_porfin), exist_ok=True)
            porfin_caus.to_excel(ruta_salida_porfin, index=False)
            print(f"✅ porfin causaciones: {ruta_salida_porfin}")

        else:
            print("⚠ No se encontraron archivos de causaciones")

    except Exception as e:
        print(f"❌ Causaciones: {e}")

    # ============================================================
    # Merge fondos + Porfin
    # ============================================================
    if porfin_caus is not None:
        try:
            fondos = pd.read_excel(RUTA_FONDOS_MITRA)
            fondos.columns = fondos.columns.astype(str).str.strip().str.replace(r"\s+", " ", regex=True)

            col_fondos_cartera = _find_col(fondos, ["NIVEL 2 - CARTERA", "NÍVEL 2 - CARTERA", "NÍVEL  2 - CARTERA"])
            col_fondos_nombre   = _find_col(fondos, ["Nombre Porfin", "NOMBRE PORFIN"])

            col_p1_cartera = _find_col(p1, ["NIVEL 2 - CARTERA", "NÍVEL 2 - CARTERA", "NÍVEL  2 - CARTERA"])
            col_p1_titulo   = _find_col(p1, ["TITULO", "Título", "TÍTULO"])
            col_p1_isin     = _find_col(p1, ["CODIGO ISIN CONTRATO", "CÓDIGO ISIN CONTRATO", "ISIN"])

            # Mapa cartera -> nombre porfin
            fondos_tmp = fondos[[col_fondos_cartera, col_fondos_nombre]].copy()
            fondos_tmp[col_fondos_cartera] = fondos_tmp[col_fondos_cartera].astype(str).str.strip()
            fondos_tmp = fondos_tmp.dropna(subset=[col_fondos_cartera])
            fondos_tmp = fondos_tmp.drop_duplicates(subset=[col_fondos_cartera], keep="first")
            mapa_nombre_porfin = fondos_tmp.set_index(col_fondos_cartera)[col_fondos_nombre]

            p1["Nombre Porfin"] = p1[col_p1_cartera].astype(str).str.strip().map(mapa_nombre_porfin)

            # Cruce por título, usando mapa para no duplicar filas
            p1["TITULO_clean"] = p1[col_p1_titulo].apply(_norm_txt).map(_norm_key)
            porfin_caus["TITULO_clean"] = porfin_caus[col_caus_titulo].apply(_norm_txt).map(_norm_key)

            mapa_titulo = porfin_caus.groupby("TITULO_clean")[col_valor_caus].sum()

            p1["VALORPORFIN"] = p1["TITULO_clean"].map(mapa_titulo)

            # Fallback ISIN + fondo
            col_caus_fondo = None
            for cand in ["Por", "POR", "Nombre Porfin", "NOMBRE PORFIN", "FONDO"]:
                try:
                    col_caus_fondo = _find_col(porfin_caus, [cand])
                    break
                except:
                    pass

            if col_caus_fondo is None:
                raise KeyError("No encontré columna de fondo en porfin_caus para el fallback ISIN + fondo.")

            porfin_caus["CLAVE_ALT"] = (
                porfin_caus["ISIN"].astype(str).map(_norm_txt).map(_norm_key) + "_" +
                porfin_caus[col_caus_fondo].astype(str).map(_norm_txt).str[:2].map(_norm_key)
            )

            p1["clave_alt"] = (
                p1[col_p1_isin].astype(str).map(_norm_txt).map(_norm_key) + "_" +
                p1["Nombre Porfin"].astype(str).map(_norm_txt).str[:2].map(_norm_key)
            )

            mapa_clave = porfin_caus.groupby("CLAVE_ALT")[col_valor_caus].sum()

            mask = p1["VALORPORFIN"].isna() | (p1["VALORPORFIN"] == 0)
            p1.loc[mask, "VALORPORFIN"] = p1.loc[mask, "clave_alt"].map(mapa_clave)

            p1.drop(columns=["clave_alt", "TITULO_clean"], inplace=True, errors="ignore")

            p1["DI - DF"] = pd.to_numeric(p1.get("DI - DF"), errors="coerce")
            p1["VALORPORFIN"] = pd.to_numeric(p1["VALORPORFIN"], errors="coerce")

            p1["DIF_CAUSACION_PORFIN"] = np.where(
                p1["VALORPORFIN"].isna(),
                np.nan,
                (p1["DI - DF"] - p1["VALORPORFIN"]).round(3)
            )

            print("✅ Merge fondos + porfin completado")

        except Exception as e:
            print(f"❌ Merge fondos: {e}")


In [ ]:
#  CELDA 14 — MITRA: Guardar resultado final
# ================================================================
if not PROCESAR_MITRA:
    pass
else:
    nombre_base = os.path.splitext(os.path.basename(RUTA_ARCHIVO_MITRA))[0]
    salida_mitra = os.path.join(carpeta_mrevision(HOY), f"{nombre_base}_VALORACION_{HOY}.xlsx")
    p1.to_excel(salida_mitra, index=False)
    print(f"✅ Mitra guardado: {salida_mitra}")

In [ ]:
#  CELDA 15 — RESUMEN FINAL
# ================================================================
print("\n" + "="*60)
print(f"  RESUMEN  |  {HOY}")
print("="*60)

if PROCESAR_PORFIN and 'df_porfin' in dir():
    print(f"\n🟢 PORFIN")
    print(f"   Filas          : {len(df_porfin):>10,}")
    print(f"   Con PRECIO_T   : {df_porfin['PRECIO_T'].notna().sum():>10,}")
    print(f"   Con VALORACION : {df_porfin['VALORACION'].notna().sum():>10,}")

if PROCESAR_MITRA and 'p1' in dir():
    print(f"\n🔵 MITRA")
    print(f"   Filas           : {len(p1):>10,}")
    if 'PRECIO_T' in p1.columns:
        print(f"   Con PRECIO_T    : {p1['PRECIO_T'].notna().sum():>10,}")
    if 'FUENTE_PRECIO' in p1.columns:
        print("\n   Por fuente:")
        print(p1['FUENTE_PRECIO'].value_counts().to_string())

print("\n" + "="*60)